# rfscorer Basic Usage

This notebook demonstrates the basic workflow of `rfscorer`:

1. Load an interaction log and split into train / test sets
2. Fit the scorer to estimate empirical revisit probabilities
3. Optimize probabilities under RF monotonicity constraints
4. Generate recommendation scores with `transform()`
5. Evaluate recommendation quality with `evaluate()`

In [ ]:
import pandas as pd
from IPython.display import Image
from rfscorer import RecencyFrequencyScorer

## 1. Load Data

The interaction log has three columns: `user_id`, `item_id`, and `date`.
Each row represents one user–item interaction.
The same pair may appear multiple times (repeat visits).

> `access_log.csv` is sourced from [ohmsha/PyOptBook](https://github.com/ohmsha/PyOptBook/tree/main/7.recommendation) (MIT License).

In [ ]:
df = pd.read_csv('access_log.csv')
print(f"{len(df):,} rows  |  {df.user_id.nunique():,} users  |  {df.item_id.nunique():,} items")
df.head()

## 2. Train / Test Split

Split users into a training set (80 %) and a test set (20 %) using a hash-based partition.

In [ ]:
df_train = df[df.user_id.map(lambda x: hash(x) % 10 < 8)]
df_test  = df[df.user_id.map(lambda x: hash(x) % 10 >= 8)]
print(f"Train: {len(df_train):,} rows ({df_train.user_id.nunique():,} users)")
print(f"Test:  {len(df_test):,} rows ({df_test.user_id.nunique():,} users)")

## 3. Fit — Empirical Revisit Probabilities

Recency and frequency are computed from the **observation period**.
Whether a user revisited an item is determined from the **evaluation period**.

In [ ]:
scorer = RecencyFrequencyScorer(user_col='user_id', item_col='item_id', datetime_col='date')

observation_period = ('2015-07-01', '2015-07-07')
evaluation_period  = ('2015-07-08', '2015-07-08')

scorer.fit(df_train, observation_period, evaluation_period)
scorer.show()

In [ ]:
scorer.plot_probability_surface(kind='empirical')
Image('surface_empirical_probability.png')

## 4. Optimize — Monotonicity Constraints (mono)

`kind='mono'` enforces that probabilities are monotone decreasing in recency
and monotone increasing in frequency.

In [ ]:
scorer.optimize(kind='mono')
scorer.plot_probability_surface(kind='mono')
Image('surface_mono_probability.png')

## 5. Optimize — MCC Constraints (mcc)

`kind='mcc'` additionally imposes convexity in recency and concavity in frequency
(diminishing marginal returns), yielding a smoother surface.

In [ ]:
scorer.optimize(kind='mcc')
scorer.plot_probability_surface(kind='mcc')
Image('surface_mcc_probability.png')

## 6. Transform — Generate Recommendation Scores

`transform()` scores each user–item pair in the test log.
Within each user, rows are sorted by `probability` descending;
`order` represents the recommendation rank (1 = top recommendation).

In [ ]:
target_date = '2015-07-07'

df_rec = scorer.transform(
    df_test, target_date, kind='mcc',
    user_col='user_id', item_col='item_id', datetime_col='date',
)
df_rec.head(10)

## 7. Evaluate — Recommendation Quality

Ground-truth revisits are interactions in `df_test` after `target_date`.
`evaluate()` reports precision, recall, and F1 at each recommendation rank cutoff.

In [ ]:
df_test_eval = df_test[df_test.date > target_date]
UIrevisit = set(zip(df_test_eval.user_id, df_test_eval.item_id))
print(f"Ground-truth revisits: {len(UIrevisit):,} user–item pairs")

scorer.evaluate(df_rec, UIrevisit, order=5)